# MortgageIQ — Full RAG Pipeline with Intelligent UI (v2.1)

A capstone RAG chatbot for mortgage documents with **production-grade guardrails**.

| Component | Implementation |
|---|---|
| PDF Extraction | PyMuPDF + pytesseract OCR fallback |
| Table Extraction | PyMuPDF `find_tables()` → structured markdown tables |
| Boundary Detection | Phi-3 Mini (anchor page strategy + keyword pre-check) |
| Chunking | Structure-aware: tables kept intact, section-based splits |
| Embeddings | BGE-small-en-v1.5 (SentenceTransformers) |
| Vector Store | FAISS (with save/load persistence) |
| Query Routing | Phi-3 predicts doc_type before retrieval |
| Answer Generation | Phi-3 Mini with structured-data-aware grounded prompts |
| Relevance Filtering | Cosine similarity threshold (configurable, default 0.35) |
| Evaluation | Per-query metrics: latency, confidence, chunk relevance stats |
| Error Handling | Try/except at every LLM call, graceful degradation |
| UI | Gradio Blocks — polished dark theme, 4-tab layout with analytics |

## Step 1: Install Dependencies

In [ ]:
!pip install -q torch
!pip install -q gradio pymupdf pypdf2 pytesseract Pillow
!pip install -q sentence-transformers faiss-cpu
!pip install -q numpy pandas
!pip install -q llama-index llama-index-embeddings-huggingface

# OCR support for scanned PDFs
!apt-get install -q tesseract-ocr

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Step 2: Download & Load Phi-3 Mini

Phi-3 Mini 4K Instruct (GGUF Q4, ~2.2 GB). `n_gpu_layers=-1` — all layers on GPU.
Used for: document boundary detection, doc_type classification, query routing, answer generation.

In [ ]:
!nvcc --version
!pip install --no-cache-dir llama-cpp-python==0.2.90 \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

In [ ]:
from llama_cpp import Llama
import os

model_path = "/content/phi3-mini.gguf"

if not os.path.exists(model_path):
    print("Downloading Phi-3 Mini GGUF (~2.2 GB)...")
    !wget -q https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-q4.gguf \
        -O {model_path}
    print(f"Downloaded to {model_path}")
else:
    print("Model already exists, skipping download.")

print(f"Model: {os.path.getsize(model_path) / (1024**2):.0f} MB")

try:
    llm_raw = Llama(
        model_path=model_path,
        n_gpu_layers=-1,
        n_ctx=4096,
        verbose=False
    )
    print("Phi-3 Mini loaded with full GPU offload!")
except Exception as e:
    print(f"Error: {e}")

## Step 3: Core Imports, Data Structures & Evaluation Logger

In [ ]:
import gradio as gr
import fitz  # PyMuPDF
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
import json
from datetime import datetime
import hashlib
import re
import time
import logging

# ── Logging setup ─────────────────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("MortgageIQ")

# ── Phi-3 helper with error handling ──────────────────────────────────────────
def phi3(prompt: str, max_tokens: int = 50) -> str:
    """Call Phi-3 Mini with correct instruction format and error handling."""
    formatted = f"<|user|>\n{prompt}<|end|>\n<|assistant|>"
    try:
        response = llm_raw(
            formatted,
            max_tokens=max_tokens,
            temperature=0.1,
            echo=False,
            stop=["<|end|>", "<|user|>"]
        )
        return response["choices"][0]["text"].strip()
    except Exception as e:
        logger.error(f"Phi-3 inference failed: {e}")
        return ""

# ── Data classes ──────────────────────────────────────────────────────────────
@dataclass
class PageInfo:
    page_num: int
    text: str
    doc_type: Optional[str] = None
    page_in_doc: int = 0

@dataclass
class LogicalDocument:
    doc_id: str
    doc_type: str
    page_start: int
    page_end: int
    text: str
    chunks: List[Dict] = field(default_factory=list)

@dataclass
class ChunkMetadata:
    chunk_id: str
    doc_id: str
    doc_type: str
    chunk_index: int
    page_start: int
    page_end: int
    text: str
    embedding: Optional[np.ndarray] = None

# ── Evaluation / metrics logger ───────────────────────────────────────────────
class QueryMetricsLogger:
    """Tracks per-query performance metrics for evaluation."""

    def __init__(self):
        self.history: List[Dict] = []

    def log(self, query, predicted_type, num_retrieved,
            avg_score, above_threshold, total_candidates,
            latency_ms, answer_length):
        entry = {
            "timestamp": datetime.now().isoformat(),
            "query": query,
            "predicted_doc_type": predicted_type,
            "num_retrieved": num_retrieved,
            "avg_relevance_score": round(avg_score, 4),
            "chunks_above_threshold": above_threshold,
            "total_candidates": total_candidates,
            "latency_ms": round(latency_ms, 1),
            "answer_length": answer_length,
        }
        self.history.append(entry)
        logger.info(f"Query metric: score={avg_score:.3f}, latency={latency_ms:.0f}ms, chunks={num_retrieved}")

    def summary(self) -> Dict:
        if not self.history:
            return {"total_queries": 0}
        scores = [h["avg_relevance_score"] for h in self.history]
        latencies = [h["latency_ms"] for h in self.history]
        return {
            "total_queries": len(self.history),
            "avg_relevance": round(sum(scores) / len(scores), 4),
            "avg_latency_ms": round(sum(latencies) / len(latencies), 1),
            "min_relevance": round(min(scores), 4),
            "max_relevance": round(max(scores), 4),
            "low_confidence_queries": sum(1 for s in scores if s < 0.35),
        }

metrics_logger = QueryMetricsLogger()
print("Imports, data classes, and metrics logger ready.")

## Step 4: Document Intelligence — Classification & Boundary Detection

Uses Phi-3 with the **anchor page strategy** — compare each page against the first page
of the current document for reliable boundary detection.

**v2:** Error handling on every LLM call, keyword pre-check for boundary detection.

In [ ]:
DOC_TYPES = [
    "Mortgage Contract", "Lender Fee Sheet", "Contract",
    "Pay Slip", "Resume", "Bank Statement", "Tax Document",
    "Insurance", "Land Deed", "Invoice", "Other"
]

def classify_document_type(text: str, max_length: int = 600) -> str:
    """Classify document type using Phi-3. Falls back to 'Other' on error."""
    sample = text[:max_length]
    prompt = f"""Classify this document into ONE of these types:
- Mortgage Contract -> home loan agreement, mortgage terms, property financing
- Lender Fee Sheet  -> loan fees, closing costs, origination charges, fees worksheet
- Contract         -> employment agreement, service agreement, legal terms
- Pay Slip         -> salary, earnings, deductions, net pay
- Resume           -> CV, work history, skills, education
- Bank Statement   -> account balance, transactions, deposits
- Tax Document     -> W2, 1099, tax return, tax form
- Insurance        -> policy, coverage, premium
- Land Deed        -> property deed, title, ownership
- Invoice          -> bill, payment request, charges
- Other            -> anything else

Document content:
{sample}

Respond with ONLY the exact label. No explanation."""

    response = phi3(prompt, max_tokens=15).strip().rstrip(".")
    if not response:
        logger.warning("Classification returned empty -- defaulting to 'Other'")
        return "Other"
    for label in DOC_TYPES:
        if label.lower() in response.lower():
            return label
    return "Other"


def is_new_document(anchor_text: str, curr_text: str, doc_type: str = None) -> bool:
    """Detect document boundaries: keyword pre-check first, then Phi-3 fallback."""
    import re

    def extract_field(text, pattern):
        m = re.search(pattern, text, re.IGNORECASE)
        return m.group(1).strip().lower() if m else None

    field_patterns = [
        r'Employee Name\s*[:\-]\s*(.+)',
        r'Employee ID\s*[:\-]\s*(\S+)',
        r'Pay Date\s*[:\-]\s*(\S+)',
        r'Application No\s*[:\-]\s*(\S+)',
        r'Account Number\s*[:\-]\s*(\S+)',
        r'Invoice No\s*[:\-]?\s*(\S+)',
        r'Policy No\s*[:\-]\s*(\S+)',
    ]

    for pattern in field_patterns:
        anchor_val = extract_field(anchor_text, pattern)
        curr_val   = extract_field(curr_text,   pattern)
        if anchor_val and curr_val and anchor_val != curr_val:
            logger.info(f"  Keyword boundary: '{anchor_val}' != '{curr_val}'")
            return True

    prompt = f"""You are a document boundary detector for a multi-document PDF.

Current document type: {doc_type or 'unknown'}

ANCHOR PAGE (first page of current document):
{anchor_text[:500]}

CURRENT PAGE:
{curr_text[:500]}

Signs of NEW document: different type, new header/letterhead, different parties/dates.
Signs of SAME document: continues numbered clauses, same parties, logical continuation.

Answer ONLY 'Yes' (new document) or 'No' (same document)."""

    response = phi3(prompt, max_tokens=10)
    if not response:
        logger.warning("Boundary detection LLM returned empty -- assuming same document")
        return False
    return "yes" in response.strip().lower()[:10]


print("Document intelligence functions ready.")

## Step 5: PDF Extraction with Table Structure Preservation

PyMuPDF extracts text from digital PDFs. For scanned pages, pytesseract OCR is the fallback.

**v2.1 key change:** Uses `page.find_tables()` to detect tabular regions on each page.
Tables are converted to explicit **markdown format** with headers and row labels preserved,
so the LLM can see which values belong to which column (e.g. Earnings vs Deductions).
Non-table text is extracted normally and combined with the structured table output.

In [ ]:
def extract_tables_as_markdown(page) -> str:
    """
    Extract tables from a PyMuPDF page and convert to markdown.
    Preserves column/row structure that flat text extraction destroys.
    """
    try:
        tables = page.find_tables()
        if not tables or len(tables.tables) == 0:
            return ""

        md_parts = []
        for table in tables.tables:
            extracted = table.extract()
            if not extracted or len(extracted) < 2:
                continue

            cleaned = []
            for row in extracted:
                cleaned.append([str(cell).strip() if cell else "" for cell in row])

            header = cleaned[0]
            # Skip tables where header is all empty (likely a false positive)
            if all(not h for h in header):
                continue

            md = "| " + " | ".join(header) + " |\n"
            md += "| " + " | ".join(["---"] * len(header)) + " |\n"
            for row in cleaned[1:]:
                while len(row) < len(header):
                    row.append("")
                md += "| " + " | ".join(row[:len(header)]) + " |\n"

            md_parts.append(md.strip())

        return "\n\n".join(md_parts)
    except Exception as e:
        logger.warning(f"Table extraction failed: {e}")
        return ""


def parse_fee_lines(text: str) -> str:
    """
    Fallback for PDFs where find_tables() fails (e.g. Calyx forms).
    Scans line-by-line for patterns like:
      Fee Name    Paid To    Paid By    $ Amount
    Groups them under section headers (ORIGINATION CHARGES, OTHER CHARGES, etc.)
    and returns a structured markdown representation.
    """
    import re
    lines = text.split("\n")
    fee_entries = []
    current_section = ""

    # Pattern: line contains a dollar amount like $ 550.00 or $550.00
    dollar_pattern = re.compile(r'\$\s*([\d,]+\.\d{2})')
    # Section headers in fee sheets
    section_pattern = re.compile(r'^(ORIGINATION CHARGES|OTHER CHARGES|GOVERNMENT CHARGES|'
                                  r'ADDITIONAL CHARGES|PREPAIDS?|ESCROW|TITLE CHARGES)',
                                  re.IGNORECASE)

    for line in lines:
        line_stripped = line.strip()
        if not line_stripped:
            continue

        # Check for section headers
        sec_match = section_pattern.match(line_stripped)
        if sec_match:
            current_section = sec_match.group(1).upper()
            continue

        # Check for fee lines (contains $ amount)
        dollar_matches = dollar_pattern.findall(line_stripped)
        if dollar_matches:
            amount = dollar_matches[-1]  # Take last dollar amount on the line

            # Extract the fee description (text before the first known entity)
            # Common paid-to entities
            entities = ["XYZ Lender", "Settlement Agent", "NOTARY", "PEST CONTROL",
                       "HI COMPANY", "Borrower", "Seller", "Lender"]
            fee_name = line_stripped
            paid_to = ""
            paid_by = ""

            for entity in entities:
                if entity.lower() in line_stripped.lower():
                    idx = line_stripped.lower().index(entity.lower())
                    if not paid_to:
                        fee_name = line_stripped[:idx].strip()
                        paid_to = entity
                    elif not paid_by:
                        paid_by = entity

            # Clean up fee name
            fee_name = re.sub(r'\$.*', '', fee_name).strip()
            fee_name = fee_name.rstrip('$').strip()

            if fee_name and len(fee_name) > 2:
                fee_entries.append({
                    "section": current_section,
                    "fee": fee_name,
                    "paid_to": paid_to,
                    "paid_by": paid_by,
                    "amount": f"${amount}"
                })

    if not fee_entries:
        return ""

    # Build structured markdown
    md = "| Fee | Paid To | Paid By | Amount |\n"
    md += "| --- | --- | --- | --- |\n"

    last_section = ""
    for entry in fee_entries:
        if entry["section"] and entry["section"] != last_section:
            md += f"| **{entry['section']}** | | | |\n"
            last_section = entry["section"]
        md += f"| {entry['fee']} | {entry['paid_to']} | {entry['paid_by']} | {entry['amount']} |\n"

    return md.strip()


def extract_text_with_tables(page) -> str:
    """
    Extract page content combining regular text with structured table data.
    Strategy:
    1. Try find_tables() first (works for proper grid-based tables)
    2. If find_tables() returns nothing, try line-by-line fee parsing (for Calyx forms)
    3. Raw text always included for non-tabular context
    """
    raw_text = page.get_text("text").strip()

    # Strategy 1: PyMuPDF table detection
    table_md = extract_tables_as_markdown(page)

    # Strategy 2: If no tables found, try fee line parsing
    if not table_md:
        table_md = parse_fee_lines(raw_text)
        if table_md:
            logger.info("    Used fee-line parsing fallback (no grid tables found)")

    if table_md:
        combined = raw_text + "\n\n--- STRUCTURED TABLE DATA (authoritative for tabular values) ---\n\n" + table_md
        return combined
    else:
        return raw_text


def extract_and_analyze_pdf(pdf_path: str) -> Tuple[List[PageInfo], List[LogicalDocument]]:
    """Extract text page by page with table structure preservation and OCR fallback."""
    logger.info("Extracting PDF...")
    doc = fitz.open(pdf_path)
    pages_info = []

    for i, page in enumerate(doc):
        text = extract_text_with_tables(page)

        if not text.strip():
            logger.info(f"  Page {i}: no text -- attempting OCR...")
            try:
                import pytesseract
                from PIL import Image
                import io
                pix = page.get_pixmap(dpi=200)
                img = Image.open(io.BytesIO(pix.tobytes("png")))
                text = pytesseract.image_to_string(img)
                logger.info(f"  Page {i}: OCR extracted {len(text.split())} words")
            except Exception as e:
                logger.error(f"  Page {i}: OCR failed -- {e}")
                text = ""
        else:
            has_table = "STRUCTURED TABLE DATA" in text
            logger.info(f"  Page {i}: extracted {len(text.split())} words" +
                        (" (with table structure)" if has_table else ""))

        pages_info.append(PageInfo(page_num=i, text=text))

    logger.info(f"Extracted {len(pages_info)} pages")

    # ── Classify pages and detect boundaries ──
    logger.info("Classifying document boundaries...")
    current_doc_type = None
    anchor_text = None
    doc_counter = 0
    page_in_doc = 0

    for i, page in enumerate(pages_info):
        if not page.text.strip():
            page.doc_type = current_doc_type
            page.page_in_doc = page_in_doc
            page_in_doc += 1
            continue

        if i == 0:
            current_doc_type = classify_document_type(page.text)
            anchor_text = page.text
            page_in_doc = 0
            logger.info(f"  Page {i}: NEW -> {current_doc_type}")
        else:
            new_doc = is_new_document(anchor_text, page.text, current_doc_type)
            if new_doc:
                doc_counter += 1
                current_doc_type = classify_document_type(page.text)
                anchor_text = page.text
                page_in_doc = 0
                logger.info(f"  Page {i}: NEW -> {current_doc_type}")
            else:
                page_in_doc += 1
                logger.info(f"  Page {i}: same ({current_doc_type}, p_in_doc={page_in_doc})")

        page.doc_type = current_doc_type
        page.page_in_doc = page_in_doc

    # ── Group pages into logical documents ──
    logical_docs = []
    current_type = None
    current_pages = []

    for page in pages_info:
        if page.doc_type != current_type and current_pages:
            doc_text = "\n\n".join(p.text for p in current_pages if p.text)
            logical_docs.append(LogicalDocument(
                doc_id=f"doc_{len(logical_docs):02d}",
                doc_type=current_type,
                page_start=current_pages[0].page_num,
                page_end=current_pages[-1].page_num,
                text=doc_text
            ))
            current_pages = []
        current_type = page.doc_type
        current_pages.append(page)

    if current_pages:
        doc_text = "\n\n".join(p.text for p in current_pages if p.text)
        logical_docs.append(LogicalDocument(
            doc_id=f"doc_{len(logical_docs):02d}",
            doc_type=current_type,
            page_start=current_pages[0].page_num,
            page_end=current_pages[-1].page_num,
            text=doc_text
        ))

    logger.info(f"Found {len(logical_docs)} logical document(s)")
    return pages_info, logical_docs


print("PDF extraction pipeline ready (with table + fee-line parsing).")

## Step 6: Structure-Aware Chunking

**v2.1 key change:** Instead of blindly splitting on word count (which rips tables apart),
the chunker now:
1. Splits text into **sections** using the `--- STRUCTURED TABLE DATA ---` marker
2. Keeps each table block **intact as a single chunk** (never split mid-table)
3. Falls back to sliding window only for non-table text blocks that exceed chunk_size
4. Every chunk retains doc_type, doc_id, page range, and a `has_table` flag

In [ ]:
TABLE_MARKER = "--- STRUCTURED TABLE DATA (authoritative for tabular values) ---"


def chunk_document(logical_doc: LogicalDocument,
                   chunk_size: int = 512,
                   overlap: int = 100) -> List[ChunkMetadata]:
    """
    Structure-aware chunking:
    - Table blocks are kept intact as single chunks (never split mid-row)
    - Non-table text uses sliding window as before
    - Small docs (< chunk_size words total) become a single chunk
    """
    full_text = logical_doc.text
    chunks = []
    chunk_counter = 0

    def make_chunk(text, idx):
        return ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_{idx}",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=idx,
            page_start=logical_doc.page_start,
            page_end=logical_doc.page_end,
            text=text.strip()
        )

    # Split on table marker to separate structured vs unstructured content
    if TABLE_MARKER in full_text:
        parts = full_text.split(TABLE_MARKER)
        for part_idx, part in enumerate(parts):
            part = part.strip()
            if not part:
                continue

            is_table_section = part_idx > 0  # parts after the marker are table data

            if is_table_section:
                # TABLE BLOCK: Keep entire table intact as one chunk
                # Prepend a label so the LLM knows this is structured data
                table_chunk_text = "STRUCTURED TABLE:\n" + part
                chunks.append(make_chunk(table_chunk_text, chunk_counter))
                chunk_counter += 1
                logger.info(f"    Table chunk kept intact ({len(part.split())} words)")
            else:
                # NON-TABLE TEXT: sliding window as before
                words = part.split()
                if len(words) <= chunk_size:
                    if words:
                        chunks.append(make_chunk(part, chunk_counter))
                        chunk_counter += 1
                else:
                    stride = chunk_size - overlap
                    for start in range(0, len(words), stride):
                        end = min(start + chunk_size, len(words))
                        chunk_text = " ".join(words[start:end])
                        chunks.append(make_chunk(chunk_text, chunk_counter))
                        chunk_counter += 1
                        if end == len(words):
                            break
    else:
        # No tables detected -- standard sliding window
        words = full_text.split()
        if len(words) <= chunk_size:
            chunks.append(make_chunk(full_text, 0))
        else:
            stride = chunk_size - overlap
            for idx, start in enumerate(range(0, len(words), stride)):
                end = min(start + chunk_size, len(words))
                chunk_text = " ".join(words[start:end])
                chunks.append(make_chunk(chunk_text, idx))
                if end == len(words):
                    break

    return chunks


def chunk_all_documents(logical_docs: List[LogicalDocument]) -> List[ChunkMetadata]:
    all_chunks = []
    for doc in logical_docs:
        chunks = chunk_document(doc)
        all_chunks.extend(chunks)
        table_chunks = sum(1 for c in chunks if "STRUCTURED TABLE" in c.text)
        logger.info(f"  {doc.doc_id} ({doc.doc_type}): {len(chunks)} chunk(s) ({table_chunks} table chunks)")
    return all_chunks


print("Structure-aware chunking ready.")

## Step 7: Embeddings & FAISS Index (with Persistence + Relevance Threshold)

BGE-small-en-v1.5 embeds all chunks. FAISS stores vectors for fast similarity search.

**v2 improvements:**
- **Relevance threshold** — chunks below configurable cosine similarity are dropped
- **FAISS persistence** — save/load index to disk so you don't re-embed on restart

In [ ]:
class IntelligentRetriever:
    """Embedding, indexing, retrieval with metadata filtering,
    relevance threshold, and FAISS persistence."""

    DEFAULT_RELEVANCE_THRESHOLD = 0.35

    def __init__(self, model_name: str = "BAAI/bge-small-en-v1.5"):
        logger.info(f"Loading embedding model: {model_name}")
        self.embed_model = SentenceTransformer(model_name)
        self.index = None
        self.chunks: List[ChunkMetadata] = []
        self.relevance_threshold = self.DEFAULT_RELEVANCE_THRESHOLD
        logger.info("Embedding model loaded.")

    def build_index(self, chunks: List[ChunkMetadata]):
        self.chunks = chunks
        texts = [c.text for c in chunks]
        logger.info(f"Embedding {len(texts)} chunks...")
        embeddings = self.embed_model.encode(texts, show_progress_bar=True, batch_size=32)

        for chunk, emb in zip(self.chunks, embeddings):
            chunk.embedding = emb

        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings.astype(np.float32))
        logger.info(f"FAISS index built -- {self.index.ntotal} vectors, dim={dim}")

    def save_index(self, path: str = "/content/faiss_index"):
        """Persist FAISS index + chunk metadata to disk."""
        if self.index is None:
            logger.warning("No index to save.")
            return
        os.makedirs(path, exist_ok=True)
        faiss.write_index(self.index, os.path.join(path, "index.faiss"))
        chunks_data = []
        for c in self.chunks:
            chunks_data.append({
                "chunk_id": c.chunk_id, "doc_id": c.doc_id,
                "doc_type": c.doc_type, "chunk_index": c.chunk_index,
                "page_start": c.page_start, "page_end": c.page_end,
                "text": c.text
            })
        with open(os.path.join(path, "chunks.json"), "w") as f:
            json.dump(chunks_data, f)
        logger.info(f"Index saved to {path}")

    def load_index(self, path: str = "/content/faiss_index") -> bool:
        """Load persisted FAISS index + chunk metadata."""
        idx_path = os.path.join(path, "index.faiss")
        chunks_path = os.path.join(path, "chunks.json")
        if not os.path.exists(idx_path) or not os.path.exists(chunks_path):
            return False
        self.index = faiss.read_index(idx_path)
        with open(chunks_path) as f:
            chunks_data = json.load(f)
        self.chunks = [ChunkMetadata(**cd, embedding=None) for cd in chunks_data]
        logger.info(f"Index loaded from {path} -- {self.index.ntotal} vectors")
        return True

    def retrieve(self, query: str, top_k: int = 5,
                 doc_type_filter: Optional[str] = None,
                 threshold: Optional[float] = None) -> List[Tuple[ChunkMetadata, float]]:
        if self.index is None or not self.chunks:
            return []

        threshold = threshold or self.relevance_threshold
        query_emb = self.embed_model.encode([query])
        faiss.normalize_L2(query_emb)

        k = top_k * 3 if doc_type_filter and doc_type_filter != "All" else top_k
        k = min(k, self.index.ntotal)

        scores, indices = self.index.search(query_emb.astype(np.float32), k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue
            if float(score) < threshold:
                continue
            chunk = self.chunks[idx]
            if doc_type_filter and doc_type_filter != "All":
                if chunk.doc_type.lower() != doc_type_filter.lower():
                    continue
            results.append((chunk, float(score)))
            if len(results) == top_k:
                break

        return results


retriever = IntelligentRetriever()
print("Retriever ready (with persistence + relevance threshold).")

## Step 8: Query Routing & Answer Generation (with Hallucination Guard)

Phi-3 predicts which doc_type the query targets before retrieval.

**v2 improvements:**
- **Hallucination guard** — post-generation check verifying answer terms exist in context
- **Confidence calibration** — low-score retrievals get explicit uncertainty flags
- **Graceful degradation** — never crashes, always returns structured response

In [ ]:
def predict_query_doc_type(query: str) -> Tuple[str, float]:
    """Predict which doc_type a query targets."""
    prompt = f"""Route this user query to the most relevant document type.

Query: "{query}"

Choose ONE from:
- Pay Slip       -> salary, earnings, deductions, net pay
- Mortgage Contract -> home loan, mortgage terms, interest rate, loan amount
- Lender Fee Sheet  -> closing costs, origination fees, lender charges
- Contract          -> employment agreement, service agreement, legal terms
- Resume            -> work history, skills, education
- Bank Statement    -> account balance, transactions, deposits
- Tax Document      -> W2, 1099, tax return
- Insurance         -> policy, coverage, premium
- Land Deed         -> property deed, title, ownership
- Invoice           -> bill, payment request
- All               -> general question, spans multiple documents

Respond with ONLY the label name."""

    response = phi3(prompt, max_tokens=10).strip().rstrip(".")
    if not response:
        return "All", 0.50
    valid = DOC_TYPES + ["All"]
    for label in valid:
        if label.lower() in response.lower():
            return label, 0.90
    return "All", 0.50


def hallucination_check(answer: str, context: str) -> Tuple[bool, float]:
    """
    Simple hallucination guard: checks what fraction of non-stopword
    content words in the answer appear somewhere in the context.
    """
    STOP_WORDS = {
        "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
        "have", "has", "had", "do", "does", "did", "will", "would", "could",
        "should", "may", "might", "shall", "can", "need", "to", "of",
        "in", "for", "on", "with", "at", "by", "from", "as", "into", "through",
        "during", "before", "after", "above", "below", "between", "out", "off",
        "over", "under", "again", "then", "once", "here", "there",
        "when", "where", "why", "how", "all", "each", "every", "both", "few",
        "more", "most", "other", "some", "such", "no", "nor", "not", "only",
        "own", "same", "so", "than", "too", "very", "just", "because", "but",
        "and", "or", "if", "while", "that", "this", "it", "its", "i", "you",
        "he", "she", "we", "they", "them", "their", "what", "which", "who",
        "these", "those", "am", "about", "also", "document", "found",
        "based", "according", "page", "pages", "source", "information",
    }
    answer_words = set(re.findall(r'[a-z0-9]+', answer.lower())) - STOP_WORDS
    context_lower = context.lower()

    if not answer_words:
        return True, 1.0

    grounded = sum(1 for w in answer_words if w in context_lower)
    ratio = grounded / len(answer_words)
    return ratio >= 0.5, round(ratio, 3)


def estimate_required_tokens(query: str, context: str) -> int:
    """
    Estimate how many tokens the answer needs based on query type and context size.
    List-type queries (what fees, what deductions, list all) need more tokens.
    """
    list_keywords = ["what all", "list", "what fees", "what deductions",
                     "how many", "all the", "every", "each", "enumerate",
                     "what are the", "what charges", "what costs", "summarize"]
    query_lower = query.lower()

    is_list_query = any(kw in query_lower for kw in list_keywords)

    # Count how many data rows might need to be listed
    # (rough heuristic: count lines with $ in context)
    dollar_lines = sum(1 for line in context.split("\n") if "$" in line)

    if is_list_query and dollar_lines > 5:
        return 600  # Long list expected
    elif is_list_query or dollar_lines > 3:
        return 450  # Medium list
    else:
        return 300  # Standard answer


def generate_answer(query: str,
                    retrieved: List[Tuple[ChunkMetadata, float]]) -> Dict:
    """Generate grounded answer with table-aware prompting, dynamic token budget,
    and hallucination check."""
    if not retrieved:
        return {
            "answer": "I couldn't find relevant information for this query in the uploaded documents.",
            "sources": [], "confidence": 0.0, "chunks_used": 0,
            "grounded": True, "grounding_ratio": 1.0,
        }

    # Use up to 5 chunks for context (not just 3) — fee sheets can be large
    max_context_chunks = 5
    context_parts = []
    sources = []
    has_tables = False
    for chunk, score in retrieved[:max_context_chunks]:
        context_parts.append(
            f"[Source: {chunk.doc_type}, Pages {chunk.page_start}–{chunk.page_end}]\n{chunk.text}"
        )
        if "STRUCTURED TABLE" in chunk.text:
            has_tables = True
        sources.append({
            "doc_type": chunk.doc_type, "doc_id": chunk.doc_id,
            "pages": f"{chunk.page_start}–{chunk.page_end}",
            "score": f"{score:.2%}", "score_raw": score,
            "preview": chunk.text[:120] + "..."
        })

    context = "\n\n".join(context_parts)
    avg_score = sum(s for _, s in retrieved[:max_context_chunks]) / min(max_context_chunks, len(retrieved))

    confidence_note = ""
    if avg_score < 0.40:
        confidence_note = "\nIMPORTANT: The retrieved context has LOW relevance. If you cannot answer confidently, say so."

    table_instructions = ""
    if has_tables:
        table_instructions = """
CRITICAL: The context contains STRUCTURED TABLES with column headers.
- READ THE COLUMN HEADERS to determine which category each value belongs to.
- The 'Paid To' column shows WHO receives the fee. Use this to filter when asked about a specific entity.
- The 'Paid By' column shows WHO pays the fee.
- List ALL matching rows from the table — do not stop after 3 items.
- For pay slips: values under 'Earnings' are earnings, values under 'Deductions' are deductions. Never mix columns."""

    # Dynamic token budget based on query complexity
    max_tokens = estimate_required_tokens(query, context)

    prompt = f"""You are a mortgage document assistant. Answer using ONLY the context below.
Rules:
- Never perform calculations -- quote exact values from the document
- Always cite the source document type and page range
- If exact answer isn't found but related info exists, provide that and explain
- Only say 'Not found' if the topic is completely absent
- When listing items, list ALL matching items from the context — never truncate{table_instructions}{confidence_note}

Context:
{context}

Question: {query}

Provide a clear, specific answer with source citation. List ALL relevant items."""

    answer = phi3(prompt, max_tokens=max_tokens)
    if not answer:
        answer = "The model failed to generate a response. Please try rephrasing your question."

    is_grounded, grounding_ratio = hallucination_check(answer, context)

    if not is_grounded:
        answer += "\n\n⚠️ **Low grounding warning:** Some parts of this answer may not be directly supported by the retrieved documents. Please verify against the source pages."
        logger.warning(f"Hallucination flag: grounding_ratio={grounding_ratio}")

    return {
        "answer": answer, "sources": sources,
        "confidence": round(avg_score, 4), "chunks_used": len(retrieved),
        "grounded": is_grounded, "grounding_ratio": grounding_ratio,
    }


print("Query routing and answer generation ready (table-aware + dynamic tokens).")

## Step 9: Document Store — Orchestrates the Full Pipeline

**v2:** FAISS persistence (save/load), metrics logging per query, timing instrumentation.

In [ ]:
class MortgageDocumentStore:
    """Orchestrates the full RAG pipeline from PDF to answer."""

    def __init__(self):
        self.pages_info:    List[PageInfo]        = []
        self.logical_docs:  List[LogicalDocument]  = []
        self.chunks:        List[ChunkMetadata]    = []
        self.is_ready = False
        self.stats = {}
        self.filename = ""

    def process(self, pdf_path: str, filename: str = "document.pdf") -> Tuple[bool, Dict]:
        """Run full pipeline: extract -> classify -> chunk -> embed -> index -> persist."""
        self.is_ready = False
        self.filename = filename
        start = datetime.now()

        try:
            self.pages_info, self.logical_docs = extract_and_analyze_pdf(pdf_path)
            self.chunks = chunk_all_documents(self.logical_docs)
            retriever.build_index(self.chunks)
            retriever.save_index()

            elapsed = (datetime.now() - start).total_seconds()
            doc_types = list(set(d.doc_type for d in self.logical_docs))

            self.stats = {
                "filename": filename,
                "total_pages": len(self.pages_info),
                "logical_docs": len(self.logical_docs),
                "total_chunks": len(self.chunks),
                "doc_types": doc_types,
                "processing_time": f"{elapsed:.1f}s"
            }
            self.is_ready = True
            return True, self.stats

        except Exception as e:
            logger.error(f"Pipeline error: {e}")
            import traceback; traceback.print_exc()
            return False, {"error": str(e)}

    def query(self, question: str, top_k: int = 5,
              doc_filter: str = "Auto", threshold: float = 0.35) -> Dict:
        """Route -> retrieve -> generate -> log metrics."""
        if not self.is_ready:
            return {"answer": "Please upload and process a document first.",
                    "sources": [], "confidence": 0.0, "chunks_used": 0,
                    "grounded": True, "grounding_ratio": 1.0}

        t0 = time.time()

        if doc_filter == "Auto":
            predicted_type, _ = predict_query_doc_type(question)
        else:
            predicted_type = doc_filter

        retrieved = retriever.retrieve(question, top_k=top_k,
                                       doc_type_filter=predicted_type,
                                       threshold=threshold)
        total_candidates = len(retrieved)

        if not retrieved and predicted_type != "All":
            logger.info(f"  No results for {predicted_type}, retrying with All...")
            retrieved = retriever.retrieve(question, top_k=top_k,
                                           doc_type_filter="All",
                                           threshold=threshold)

        result = generate_answer(question, retrieved)
        result["predicted_doc_type"] = predicted_type

        latency_ms = (time.time() - t0) * 1000
        avg_score = result["confidence"]
        above_thresh = sum(1 for _, s in retrieved if s >= threshold)
        metrics_logger.log(
            query=question, predicted_type=predicted_type,
            num_retrieved=len(retrieved), avg_score=avg_score,
            above_threshold=above_thresh, total_candidates=total_candidates,
            latency_ms=latency_ms, answer_length=len(result["answer"]),
        )
        result["latency_ms"] = round(latency_ms, 1)
        return result


doc_store = MortgageDocumentStore()
print("Document store ready (with persistence + metrics).")

## Step 10: Gradio UI — Polished 4-Tab Layout

**Redesigned UI:**
- **Upload tab** — drag-and-drop PDF, processing stats, document structure cards
- **Chat tab** — conversation panel + retrieval sidebar with visual confidence bars
- **Analytics tab** — live query performance dashboard
- **About tab** — full pipeline reference with v2 changelog

In [ ]:
# ── Gradio event handlers ─────────────────────────────────────────────────────

def handle_upload(pdf_file):
    """Process uploaded PDF and return status + document structure."""
    if pdf_file is None:
        return "⏳ *Waiting for a PDF upload...*", "", gr.update(choices=["Auto"])

    filename = os.path.basename(pdf_file.name) if hasattr(pdf_file, "name") else "document.pdf"
    success, stats = doc_store.process(pdf_file.name, filename)

    if not success:
        return f"❌ **Error:** {stats.get('error', 'Unknown error')}", "", gr.update(choices=["Auto"])

    status = f"""✅ **{stats['filename']}** processed successfully!

| Metric | Value |
|:---|:---|
| 📄 Pages extracted | **{stats['total_pages']}** |
| 📂 Logical documents | **{stats['logical_docs']}** |
| 🧩 Chunks created | **{stats['total_chunks']}** |
| ⏱️ Processing time | **{stats['processing_time']}** |
| 💾 FAISS index | **Saved to disk** |
"""

    structure_lines = []
    for d in doc_store.logical_docs:
        n_chunks = len([c for c in doc_store.chunks if c.doc_id == d.doc_id])
        structure_lines.append(
            f"📌 **{d.doc_type}** — `{d.doc_id}` — Pages {d.page_start}–{d.page_end} — {n_chunks} chunks"
        )
    structure = "\n\n".join(structure_lines)

    filter_choices = ["Auto", "All"] + stats["doc_types"]
    return status, structure, gr.update(choices=filter_choices, value="Auto")


def handle_query(message, history, doc_filter, top_k, threshold):
    """Handle a chat query."""
    if not message.strip():
        return history, "*Type a question to get started.*"

    result = doc_store.query(message, top_k=int(top_k),
                              doc_filter=doc_filter,
                              threshold=float(threshold))

    answer     = result["answer"]
    conf       = result["confidence"]
    chunks     = result["chunks_used"]
    routed     = result.get("predicted_doc_type", "All")
    latency    = result.get("latency_ms", 0)
    grounded   = result.get("grounded", True)
    grounding  = result.get("grounding_ratio", 1.0)

    conf_label = "High" if conf >= 0.5 else "Medium" if conf >= 0.35 else "Low"
    ground_icon = "✅" if grounded else "⚠️"

    history.append((message, answer))

    sources_md = f"""### 📊 Retrieval Info

| Metric | Value |
|:---|:---|
| 🎯 Routed to | **{routed}** |
| 🧩 Chunks used | **{chunks}** |
| ⏱️ Latency | **{latency:.0f} ms** |
| {ground_icon} Grounding | **{grounding:.0%}** |

**Confidence: {conf_label} ({conf:.1%})**

"""

    for i, src in enumerate(result["sources"][:3], 1):
        score_val = src.get("score_raw", 0)
        bar_len = int(score_val * 20)
        bar = "█" * bar_len + "░" * (20 - bar_len)
        sources_md += f"""
**{i}. {src['doc_type']}** — Pages {src['pages']}
`{bar}` {src['score']}
_{src['preview']}_

---
"""
    return history, sources_md


def get_analytics():
    """Generate analytics markdown."""
    summary = metrics_logger.summary()
    if summary["total_queries"] == 0:
        return "📊 *No queries yet. Ask some questions first!*"

    s = summary
    md_text = f"""### 📈 Query Performance Dashboard

| Metric | Value |
|:---|:---|
| Total queries | **{s['total_queries']}** |
| Avg relevance score | **{s['avg_relevance']:.3f}** |
| Min / Max relevance | **{s['min_relevance']:.3f}** / **{s['max_relevance']:.3f}** |
| Avg latency | **{s['avg_latency_ms']:.0f} ms** |
| Low-confidence queries | **{s['low_confidence_queries']}** |

---

### 📋 Query History (last 10)

| # | Query | Doc Type | Score | Latency | Chunks |
|:---|:---|:---|:---|:---|:---|
"""
    for i, h in enumerate(metrics_logger.history[-10:], 1):
        q_short = h["query"][:40] + ("..." if len(h["query"]) > 40 else "")
        md_text += f"| {i} | {q_short} | {h['predicted_doc_type']} | {h['avg_relevance_score']:.3f} | {h['latency_ms']:.0f}ms | {h['num_retrieved']} |\n"

    return md_text


def clear_chat():
    return [], "*Sources will appear here after your first question.*"


# ── Build UI ──────────────────────────────────────────────────────────────────
CSS = """
.gradio-container {
    font-family: 'Inter', 'Segoe UI', system-ui, sans-serif !important;
    max-width: 1200px !important;
    margin: auto !important;
}
#header-banner {
    background: linear-gradient(135deg, #0f172a 0%, #1e293b 50%, #0f4c75 100%);
    border-radius: 16px;
    padding: 32px 40px;
    margin-bottom: 20px;
    border: 1px solid rgba(56, 189, 248, 0.15);
    box-shadow: 0 8px 32px rgba(0, 0, 0, 0.3);
}
#header-banner h1 {
    color: #38bdf8; margin: 0 0 6px 0;
    font-size: 2.2rem; font-weight: 700; letter-spacing: -0.5px;
}
#header-banner .subtitle { color: #94a3b8; font-size: 1rem; margin: 0; }
#header-banner .badges { margin-top: 12px; display: flex; gap: 8px; flex-wrap: wrap; }
#header-banner .badge {
    background: rgba(56, 189, 248, 0.1);
    border: 1px solid rgba(56, 189, 248, 0.25);
    color: #7dd3fc; padding: 4px 12px;
    border-radius: 20px; font-size: 0.75rem; font-weight: 500;
}
.source-panel {
    background: linear-gradient(180deg, #f8fafc 0%, #f1f5f9 100%) !important;
    border-radius: 12px !important; padding: 20px !important;
    border-left: 4px solid #0ea5e9 !important;
    box-shadow: 0 2px 8px rgba(0,0,0,0.04) !important;
}
.source-panel code {
    background: rgba(14, 165, 233, 0.08) !important; color: #0369a1 !important;
    font-family: 'JetBrains Mono', monospace !important;
    padding: 2px 4px !important; border-radius: 4px !important; font-size: 0.8rem !important;
}
.source-panel p, .source-panel h3, .source-panel strong, .source-panel em,
.source-panel td, .source-panel th { color: #1e293b !important; }
.source-panel table { border-collapse: collapse; width: 100%; }
.source-panel td, .source-panel th { padding: 6px 10px; }
.source-panel hr { border-color: #e2e8f0; }
.analytics-panel {
    background: linear-gradient(180deg, #f8fafc 0%, #f1f5f9 100%) !important;
    border-radius: 12px !important; padding: 20px !important;
    border-left: 4px solid #8b5cf6 !important;
}
.analytics-panel p, .analytics-panel h3, .analytics-panel strong,
.analytics-panel td, .analytics-panel th { color: #1e293b !important; }
.analytics-panel table { border-collapse: collapse; width: 100%; }
.analytics-panel td, .analytics-panel th { padding: 6px 10px; }
.primary-btn {
    background: linear-gradient(135deg, #0ea5e9 0%, #0284c7 100%) !important;
    border: none !important; color: white !important;
    font-weight: 600 !important; border-radius: 10px !important;
    padding: 10px 24px !important;
    box-shadow: 0 4px 12px rgba(14, 165, 233, 0.3) !important;
}
.tab-nav button { font-weight: 600 !important; font-size: 0.95rem !important; }
.tab-nav button.selected { border-bottom: 3px solid #0ea5e9 !important; color: #0ea5e9 !important; }
"""

HEADER_HTML = """
<div id="header-banner">
  <h1>🏠 MortgageIQ</h1>
  <p class="subtitle">Intelligent RAG pipeline for mortgage documents — Phi-3 Mini & BGE embeddings</p>
  <div class="badges">
    <span class="badge">🔒 Fully Local</span>
    <span class="badge">🧠 Phi-3 Mini 4K</span>
    <span class="badge">⚡ FAISS Vector Search</span>
    <span class="badge">📊 Hallucination Guard</span>
    <span class="badge">💾 Persistent Index</span>
    <span class="badge">📋 Table Structure</span>
  </div>
</div>
"""

with gr.Blocks(css=CSS, title="MortgageIQ — RAG Chatbot", theme=gr.themes.Soft()) as demo:

    gr.HTML(HEADER_HTML)

    with gr.Tabs():

        # ── Tab 1: Upload ──
        with gr.TabItem("📁 Upload Document"):
            gr.Markdown("Upload a **PDF** (digital or scanned). OCR is automatic. FAISS index is persisted to disk.")
            with gr.Row():
                with gr.Column(scale=2):
                    pdf_input = gr.File(label="Drop your PDF here", file_types=[".pdf"], type="filepath")
                    upload_btn = gr.Button("⚡ Process Document", variant="primary", elem_classes=["primary-btn"])
                with gr.Column(scale=3):
                    status_out = gr.Markdown(value="⏳ *No document loaded yet. Upload a PDF to begin.*")
            gr.Markdown("### 📂 Document Structure")
            structure_out = gr.Markdown(value="*Will appear after processing.*")

        # ── Tab 2: Chat ──
        with gr.TabItem("💬 Ask Questions"):
            with gr.Row():
                with gr.Column(scale=3):
                    chatbot = gr.Chatbot(label="MortgageIQ Chat", height=500,
                                         show_copy_button=True, avatar_images=(None, "🏠"),
                                         bubble_full_width=False)
                    with gr.Row():
                        msg_input = gr.Textbox(placeholder="Ask about your mortgage documents...",
                                               label="Your Question", scale=5,
                                               show_label=False, container=False)
                        send_btn = gr.Button("Send ➤", variant="primary", scale=1, elem_classes=["primary-btn"])
                    with gr.Row():
                        clear_btn = gr.Button("🗑️ Clear Chat", scale=1)
                        gr.Markdown("*Tip: Set filter to Auto for intelligent routing, or pick a specific doc type.*")

                with gr.Column(scale=2):
                    with gr.Accordion("⚙️ Retrieval Settings", open=True):
                        doc_filter = gr.Dropdown(label="Document Filter", choices=["Auto", "All"],
                                                 value="Auto", info="Auto = Phi-3 predicts best doc type")
                        top_k = gr.Slider(label="Chunks to retrieve (top-k)",
                                          minimum=1, maximum=10, value=5, step=1)
                        threshold_slider = gr.Slider(label="Relevance threshold",
                                                     minimum=0.0, maximum=1.0, value=0.35, step=0.05,
                                                     info="Chunks below this cosine similarity are dropped")
                    sources_panel = gr.Markdown(
                        value="*Sources will appear here after your first question.*",
                        elem_classes=["source-panel"])

        # ── Tab 3: Analytics ──
        with gr.TabItem("📈 Analytics"):
            gr.Markdown("### Live Query Performance Dashboard")
            gr.Markdown("Track relevance scores, latency, grounding ratios across your session.")
            refresh_btn = gr.Button("🔄 Refresh Analytics", variant="secondary")
            analytics_panel = gr.Markdown(value="📊 *No queries yet.*", elem_classes=["analytics-panel"])

        # ── Tab 4: About ──
        with gr.TabItem("ℹ️ About"):
            gr.Markdown("""
### 🏠 MortgageIQ v2.1 — Pipeline Overview

| Stage | Method | v2 Improvement |
|:---|:---|:---|
| PDF Extraction | PyMuPDF + pytesseract OCR | **Table structure preservation via find_tables()** |
| Boundary Detection | Phi-3 (anchor page + keyword) | Error handling on empty LLM responses |
| Doc Classification | Phi-3 — 11 labels | Graceful fallback to "Other" |
| Chunking | Structure-aware splits | **Tables kept intact, never split mid-row** |
| Embeddings | BGE-small-en-v1.5 | — |
| Vector Store | FAISS flat IP index | **Save/load persistence** |
| Query Routing | Phi-3 doc_type prediction | — |
| Answer Generation | Phi-3 grounded prompts | **Table-aware prompting + hallucination guard** |
| Relevance Filter | Cosine threshold (0.35) | **New — configurable in UI** |
| Evaluation | Per-query metrics | **New — Analytics dashboard** |
| Error Handling | Try/except everywhere | **New — graceful degradation** |

**Model:** Phi-3 Mini 4K Instruct (GGUF Q4, ~2.2 GB, T4 GPU)
**No API key required. Fully local.**

---

**v2.1 — Table Structure Fix:**
6. ✅ Table extraction — `find_tables()` converts PDF tables to markdown with column headers
7. ✅ Structure-aware chunking — tables kept as intact chunks, never split mid-row
8. ✅ Table-aware prompting — explicit instructions to read column headers before answering

**v2 Gaps Addressed:**
1. ✅ Evaluation metrics — per-query latency, relevance, grounding, analytics dashboard
2. ✅ Error handling — try/except on all Phi-3 calls, graceful fallback
3. ✅ Hallucination guard — word-overlap check with warning flags
4. ✅ Relevance threshold — configurable cosine cutoff
5. ✅ FAISS persistence — save/load index to disk
            """)

    # ── Wire events ──
    upload_btn.click(fn=handle_upload, inputs=pdf_input, outputs=[status_out, structure_out, doc_filter])

    send_btn.click(fn=handle_query, inputs=[msg_input, chatbot, doc_filter, top_k, threshold_slider],
                   outputs=[chatbot, sources_panel]).then(lambda: "", outputs=msg_input)

    msg_input.submit(fn=handle_query, inputs=[msg_input, chatbot, doc_filter, top_k, threshold_slider],
                     outputs=[chatbot, sources_panel]).then(lambda: "", outputs=msg_input)

    clear_btn.click(fn=clear_chat, outputs=[chatbot, sources_panel])
    refresh_btn.click(fn=get_analytics, outputs=analytics_panel)

print("UI built. Run the next cell to launch.")

In [ ]:
demo.launch(share=True, debug=False)